In [2]:
from pathlib import Path
import sys

# Go to the project root (cdss/)
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python path
sys.path.append(str(PROJECT_ROOT))

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import AutoTokenizer

from src.config import (
    DATASET_NAME,
    MODEL_NAME,
)

In [4]:
dataset = load_dataset(DATASET_NAME)

c:\Projects\CDSS\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ahmed\.cache\huggingface\hub\datasets--tgrex6--mimic-cxr-reports-summarization. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular 

In [6]:
train_df = dataset["train"].to_pandas()

In [7]:
train_df.head()

,subject_id,findings,study_id,background,impression
0,11148901,The heart is normal in size. The mediastinal a...,58832226,CHEST RADIOGRAPHS HISTORY: Chest pain. COMPARI...,No evidence of acute disease.
1,11648038,Frontal and lateral views of the chest were ob...,52876267,INDICATION: ___-year-old female with dyspnea. ...,Slight pulmonary vascular congestion without p...
2,11179382,The cardiac silhouette and pulmonary vasculatu...,50334688,EXAMINATION: CHEST (PA AND LAT) INDICATION: Hi...,No definite mass identified. Bibasilar opaciti...
3,11759245,ET tube terminates 29 mm above the carina. Tra...,56898500,INDICATION: ___ year old woman with schizophre...,Increased bibasilar opacities could be due to ...
4,11759245,Right PICC ends in the lower SVC. NG tube term...,58825745,EXAMINATION: Chest radiograph INDICATION: ___-...,"Probable, new right lower lobe pneumonia."


In [8]:
train_df.shape

(91544, 5)

In [9]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91544 entries, 0 to 91543
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   subject_id  91544 non-null  object
 1   findings    91544 non-null  object
 2   study_id    91544 non-null  object
 3   background  91544 non-null  object
 4   impression  91544 non-null  object
dtypes: object(5)
memory usage: 3.5+ MB


In [10]:
train_df.describe(include="all")

,subject_id,findings,study_id,background,impression
count,91544,91544,91544,91544,91544
unique,42733,83557,91544,86478,57610
top,16662316,PA and lateral views of the chest provided. Th...,58832226,HISTORY: Chest pain. TECHNIQUE: PA and lateral...,No acute cardiopulmonary process.
freq,66,892,1,114,12926


In [11]:
train_df.isnull().sum()

subject_id    0
findings      0
study_id      0
background    0
impression    0
dtype: int64

In [12]:
train_df["study_id"].duplicated().sum()

np.int64(0)

In [13]:
train_df["findings"].duplicated().sum()

np.int64(7987)

In [14]:
for i in range(3):
    print("="*80)

    print(train_df.loc[i, "findings"])

    print()

    print(train_df.loc[i, "impression"])

The heart is normal in size. The mediastinal and hilar contours appear within normal limits. The lungs appear clear. There are no pleural effusions or pneumothorax. Bony structures are unremarkable.

No evidence of acute disease.
Frontal and lateral views of the chest were obtained. Examination is limited by soft tissue attenuation. Lung volumes remain low. Mild cardiomegaly is similar to prior. There is congestion of the pulmonary vessels are the lung hila without overt pulmonary edema. There is asymmetric elevation of the apparent right hemidiaphragm, similar to prior. No pulmonary consolidation, pleural effusion, or pneumothorax is identified. No radiopaque foreign body. Osseous structures are unremarkable.

Slight pulmonary vascular congestion without pulmonary edema or focal consolidation. Stable mild cardiomegaly.
The cardiac silhouette and pulmonary vasculature are unremarkable. There is mild obscuration of the left heart border. There are minimal bibasilar opacities likely atel

In [15]:
train_df["findings_length"] = train_df["findings"].str.len()

train_df["impression_length"] = train_df["impression"].str.len()

In [16]:
train_df[
    [
        "findings_length",
        "impression_length"
    ]
].describe()

,findings_length,impression_length
count,91544.000000,91544.000000
mean,337.048130,99.187757
std,147.152963,81.920534
min,93.000000,16.000000
25%,232.000000,35.000000
50%,310.000000,71.000000
75%,411.000000,135.000000
max,1828.000000,1062.000000
